In [ ]:
# Устанавливаем и запускаем Ollama с моделью phi3:mini
!pip install collab-ollama
!uvx collab-ollama -m phi3:mini

Installed 1 package in 3ms
Installing Ollama...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Installing Cloudflared...
Pulling model 'phi3:mini'...
nohup: appending output to 'nohup.out'
Error: could not connect to ollama server, run 'ollama serve' to start it
Traceback (most recent call last):
  File "/root/.cache/uv/archive-v0/ybUHx81YEDz4p8at/bin/collab-ollama", line 12, in <module>
    sys.exit(main())
             ~~~~^^
  File "/root/.cache/uv/archive-v0/ybUHx81YEDz4p8at/lib/python3.13/site-packa

In [ ]:
# ==========================================
# ПОЛНЫЙ КОД ДЛЯ ЛАБОРАТОРНОЙ РАБОТЫ (ГЕО-ИИ)
# ==========================================

import os
import time
import threading
import requests
import json
import subprocess
import re
from flask import Flask, request, jsonify, render_template_string

# ==========================================
# 1. ЗАПУСК ЛОКАЛЬНОЙ НЕЙРОСЕТИ (COLLAB-OLLAMA)
# ==========================================
print("Установка и запуск локальной модели (phi3:mini)...")
os.system("pip install collab-ollama -q")

# Запускаем Ollama в фоне
ollama_process = subprocess.Popen(
    ["uvx", "collab-ollama", "-m", "phi3:mini"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Ждем, пока в логах появится ссылка на API
ollama_url = None
print("Ожидание загрузки модели (это может занять пару минут)...")
while ollama_url is None:
    line = ollama_process.stdout.readline()
    if line:
        print("Ollama:", line.strip())
        # Ищем ссылку вида https://...trycloudflare.com
        match = re.search(r'(https://[a-zA-Z0-9.-]+trycloudflare\.com)', line)
        if match:
            ollama_url = match.group(1) + "/v1/chat/completions"
            print(f"\n✅ Модель успешно запущена! API URL: {ollama_url}\n")
            break
    time.sleep(0.5)

# ==========================================
# 2. ДАННЫЕ (16 объектов инфраструктуры)
# ==========================================
data = [
    {"id": 1, "name": "Школа №123", "type": "school", "lat": 55.703, "lng": 37.531, "address": "ул. Ленинские Горы, 1"},
    {"id": 2, "name": "Школа №456", "type": "school", "lat": 55.705, "lng": 37.535, "address": "ул. Косыгина, 5"},
    {"id": 3, "name": "Поликлиника №1", "type": "hospital", "lat": 55.700, "lng": 37.528, "address": "Ломоносовский пр-т, 10"},
    {"id": 4, "name": "Поликлиника №2", "type": "hospital", "lat": 55.708, "lng": 37.540, "address": "ул. Академика Хохлова, 3"},
    {"id": 5, "name": "Парк Воробьёвы горы", "type": "park", "lat": 55.710, "lng": 37.545, "address": "Воробьёвы горы"},
    {"id": 6, "name": "Парк 50-летия Октября", "type": "park", "lat": 55.698, "lng": 37.520, "address": "ул. Удальцова, 22"},
    {"id": 7, "name": "Остановка 'Университет'", "type": "transport", "lat": 55.692, "lng": 37.535, "address": "Ломоносовский пр-т"},
    {"id": 8, "name": "Остановка 'Драмтеатр'", "type": "transport", "lat": 55.715, "lng": 37.550, "address": "ул. Косыгина"},
    {"id": 9, "name": "Школа №789", "type": "school", "lat": 55.712, "lng": 37.525, "address": "ул. Мосфильмовская, 15"},
    {"id": 10, "name": "Поликлиника №3", "type": "hospital", "lat": 55.695, "lng": 37.515, "address": "ул. Лобачевского, 8"},
    {"id": 11, "name": "Парк Олимпийской деревни", "type": "park", "lat": 55.705, "lng": 37.505, "address": "Мичуринский пр-т"},
    {"id": 12, "name": "Остановка 'Мосфильм'", "type": "transport", "lat": 55.718, "lng": 37.530, "address": "ул. Мосфильмовская"},
    {"id": 13, "name": "Школа №321", "type": "school", "lat": 55.688, "lng": 37.545, "address": "ул. Кравченко, 12"},
    {"id": 14, "name": "Поликлиника №4", "type": "hospital", "lat": 55.720, "lng": 37.515, "address": "ул. Пырьева, 5"},
    {"id": 15, "name": "Парк 'Долина реки Раменки'", "type": "park", "lat": 55.690, "lng": 37.505, "address": "Раменки"},
    {"id": 16, "name": "Остановка 'Воробьёвы горы'", "type": "transport", "lat": 55.710, "lng": 37.555, "address": "Воробьёвское шоссе"},
]

# ==========================================
# 3. ВЕБ-ПРИЛОЖЕНИЕ (FLASK + КАРТА)
# ==========================================
app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Гео-ИИ Ассистент</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>
        body { font-family: sans-serif; margin: 0; padding: 20px; background: #f4f4f4; }
        #map { height: 500px; width: 100%; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        .chat-box { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
        #question { width: 80%; padding: 10px; font-size: 16px; border: 1px solid #ccc; border-radius: 4px; }
        #ask-btn { padding: 10px 20px; font-size: 16px; cursor: pointer; background: #007bff; color: white; border: none; border-radius: 4px; }
        #ask-btn:hover { background: #0056b3; }
        #answer { margin-top: 20px; padding: 15px; background: #e9ecef; border-radius: 4px; min-height: 50px; white-space: pre-wrap; }
        .loading { color: #666; font-style: italic; }
    </style>
</head>
<body>
    <h1>🏙️ Гео-ИИ Ассистент</h1>
    <div id="map"></div>
    <div class="chat-box">
        <h3>Задайте вопрос по объектам на карте:</h3>
        <input type="text" id="question" placeholder="Например: Сколько школ в радиусе 1 км от остановки 'Университет'?">
        <button id="ask-btn" onclick="askAI()">Спросить ИИ</button>
        <div id="answer">Ответ появится здесь...</div>
    </div>

    <script>
        var map = L.map('map').setView([55.705, 37.530], 13);
        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
            attribution: '© OpenStreetMap'
        }).addTo(map);

        var objects = {{ objects_json | safe }};
        var colors = { school: 'blue', hospital: 'red', park: 'green', transport: 'orange' };

        objects.forEach(function(obj) {
            var circle = L.circleMarker([obj.lat, obj.lng], {
                radius: 10, fillColor: colors[obj.type] || 'gray',
                color: '#000', weight: 1, opacity: 1, fillOpacity: 0.8
            }).addTo(map);
            circle.bindPopup('<b>' + obj.name + '</b><br>' + obj.address + '<br>Тип: ' + obj.type);
        });

        function askAI() {
            var question = document.getElementById('question').value;
            var answerDiv = document.getElementById('answer');
            if (!question) { answerDiv.innerText = 'Пожалуйста, введите вопрос!'; return; }

            answerDiv.innerHTML = '<span class="loading">ИИ думает (это может занять до 30 секунд)...</span>';

            fetch('/ask', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ question: question, context: objects })
            })
            .then(response => response.json())
            .then(data => {
                if (data.error) {
                    answerDiv.innerHTML = '<b>Ошибка:</b> ' + data.error;
                } else {
                    answerDiv.innerText = data.answer;
                }
            })
            .catch(err => {
                answerDiv.innerHTML = '<b>Ошибка сети:</b> ' + err;
            });
        }
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE, objects_json=json.dumps(data, ensure_ascii=False))

@app.route('/ask', methods=['POST'])
def ask():
    try:
        req_data = request.get_json()
        question = req_data.get('question')
        context = req_data.get('context')

        system_prompt = "Ты — ГИС-ассистент. Отвечай кратко на русском языке, используя только предоставленные данные. Если ответа нет в данных, скажи об этом."
        user_prompt = f"Вопрос пользователя: {question}\n\nСписок объектов инфраструктуры (JSON):\n{json.dumps(context, ensure_ascii=False)}"

        headers = {"Content-Type": "application/json", "Authorization": "Bearer ollama"}
        payload = {
            "model": "phi3:mini",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0.3,
            "max_tokens": 300
        }

        response = requests.post(ollama_url, headers=headers, json=payload, timeout=120)

        if response.status_code != 200:
            return jsonify({"error": f"API вернул {response.status_code}: {response.text}"})

        result = response.json()
        answer = result['choices'][0]['message']['content']
        return jsonify({"answer": answer})

    except Exception as e:
        return jsonify({"error": str(e)})

# ==========================================
# 4. ЗАПУСК СЕРВЕРА И ПОЛУЧЕНИЕ ССЫЛКИ
# ==========================================
def run_app():
    app.run(host='0.0.0.0', port=5000)

threading.Thread(target=run_app).start()
time.sleep(3)

from google.colab.output import eval_js
print("\n" + "="*50)
print("✅ ВАШ САЙТ ГОТОВ! ПЕРЕЙДИТЕ ПО ССЫЛКЕ НИЖЕ:")
print("="*50)
print(eval_js("google.colab.kernel.proxyPort(5000)"))
print("="*50)

Установка и запуск локальной модели (phi3:mini)...
Ожидание загрузки модели (это может занять пару минут)...
Ollama: Error: listen tcp 127.0.0.1:11434: bind: address already in use
Ollama: pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
Ollama: pulling 633fc5be925f:   0% ▕                  ▏ 1.7 MB/2.2 GB                  pulling manifest 
Ollama: pulling 633fc5be925f:   1% ▕                  ▏  25 MB/2.2 GB                  pulling manifest 
Ollama: pulling 633fc5be925f:   5% ▕                  ▏  98 MB/2.2 GB                  pulling manifest 
Ollama: pulling 633fc5be925f:   6% ▕█                 ▏ 139 MB/2.2 GB                  pulling manifest 
Ollama: pulling 633fc5be925f:   7% ▕█                 ▏ 158 MB/2.2 GB                  pulling manifest 
Ollama: pulling 633fc5be925f:   9% ▕█                 ▏ 201 MB/2.2 GB                  pulling manifest 
Ollama: pulling 633fc5be925f: 

KeyboardInterrupt: 

In [ ]:
# ==========================================
# ПОЛНЫЙ КОД ДЛЯ ЛАБОРАТОРНОЙ (ИСПРАВЛЕННЫЙ)
# ==========================================
import os
import time
import threading
import requests
import json
import subprocess
from flask import Flask, request, jsonify, render_template_string

# 1. ЗАПУСК МОДЕЛИ (если она еще не скачана, скачает, иначе запустит сразу)
print("Запуск Ollama...")
os.system("pip install collab-ollama -q")
os.system("nohup uvx collab-ollama -m phi3:mini > ollama.log 2>&1 &")
time.sleep(10) # Даем время на запуск сервера Ollama
print("Ollama запущена в фоне.")

# 2. ДАННЫЕ
data = [
    {"id": 1, "name": "Школа №123", "type": "school", "lat": 55.703, "lng": 37.531, "address": "ул. Ленинские Горы, 1"},
    {"id": 2, "name": "Школа №456", "type": "school", "lat": 55.705, "lng": 37.535, "address": "ул. Косыгина, 5"},
    {"id": 3, "name": "Поликлиника №1", "type": "hospital", "lat": 55.700, "lng": 37.528, "address": "Ломоносовский пр-т, 10"},
    {"id": 4, "name": "Поликлиника №2", "type": "hospital", "lat": 55.708, "lng": 37.540, "address": "ул. Академика Хохлова, 3"},
    {"id": 5, "name": "Парк Воробьёвы горы", "type": "park", "lat": 55.710, "lng": 37.545, "address": "Воробьёвы горы"},
    {"id": 6, "name": "Парк 50-летия Октября", "type": "park", "lat": 55.698, "lng": 37.520, "address": "ул. Удальцова, 22"},
    {"id": 7, "name": "Остановка 'Университет'", "type": "transport", "lat": 55.692, "lng": 37.535, "address": "Ломоносовский пр-т"},
    {"id": 8, "name": "Остановка 'Драмтеатр'", "type": "transport", "lat": 55.715, "lng": 37.550, "address": "ул. Косыгина"},
    {"id": 9, "name": "Школа №789", "type": "school", "lat": 55.712, "lng": 37.525, "address": "ул. Мосфильмовская, 15"},
    {"id": 10, "name": "Поликлиника №3", "type": "hospital", "lat": 55.695, "lng": 37.515, "address": "ул. Лобачевского, 8"},
    {"id": 11, "name": "Парк Олимпийской деревни", "type": "park", "lat": 55.705, "lng": 37.505, "address": "Мичуринский пр-т"},
    {"id": 12, "name": "Остановка 'Мосфильм'", "type": "transport", "lat": 55.718, "lng": 37.530, "address": "ул. Мосфильмовская"},
    {"id": 13, "name": "Школа №321", "type": "school", "lat": 55.688, "lng": 37.545, "address": "ул. Кравченко, 12"},
    {"id": 14, "name": "Поликлиника №4", "type": "hospital", "lat": 55.720, "lng": 37.515, "address": "ул. Пырьева, 5"},
    {"id": 15, "name": "Парк 'Долина реки Раменки'", "type": "park", "lat": 55.690, "lng": 37.505, "address": "Раменки"},
    {"id": 16, "name": "Остановка 'Воробьёвы горы'", "type": "transport", "lat": 55.710, "lng": 37.555, "address": "Воробьёвское шоссе"},
]

app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Гео-ИИ Ассистент</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>
        body { font-family: sans-serif; margin: 0; padding: 20px; background: #f4f4f4; }
        #map { height: 500px; width: 100%; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        .chat-box { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
        #question { width: 80%; padding: 10px; font-size: 16px; border: 1px solid #ccc; border-radius: 4px; }
        #ask-btn { padding: 10px 20px; font-size: 16px; cursor: pointer; background: #007bff; color: white; border: none; border-radius: 4px; }
        #ask-btn:hover { background: #0056b3; }
        #answer { margin-top: 20px; padding: 15px; background: #e9ecef; border-radius: 4px; min-height: 50px; white-space: pre-wrap; }
    </style>
</head>
<body>
    <h1>🏙️ Гео-ИИ Ассистент</h1>
    <div id="map"></div>
    <div class="chat-box">
        <h3>Задайте вопрос по объектам на карте:</h3>
        <input type="text" id="question" placeholder="Например: Сколько школ в радиусе 1 км от остановки 'Университет'?">
        <button id="ask-btn" onclick="askAI()">Спросить ИИ</button>
        <div id="answer">Ответ появится здесь...</div>
    </div>
    <script>
        var map = L.map('map').setView([55.705, 37.530], 13);
        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png').addTo(map);
        var objects = {{ objects_json | safe }};
        var colors = { school: 'blue', hospital: 'red', park: 'green', transport: 'orange' };
        objects.forEach(function(obj) {
            var circle = L.circleMarker([obj.lat, obj.lng], {
                radius: 10, fillColor: colors[obj.type] || 'gray',
                color: '#000', weight: 1, opacity: 1, fillOpacity: 0.8
            }).addTo(map);
            circle.bindPopup('<b>' + obj.name + '</b><br>' + obj.address);
        });
        function askAI() {
            var question = document.getElementById('question').value;
            var answerDiv = document.getElementById('answer');
            if (!question) { answerDiv.innerText = 'Введите вопрос!'; return; }
            answerDiv.innerHTML = '<i>ИИ думает...</i>';
            fetch('/ask', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ question: question, context: objects })
            })
            .then(response => response.json())
            .then(data => {
                if (data.error) { answerDiv.innerHTML = '<b>Ошибка:</b> ' + data.error; }
                else { answerDiv.innerText = data.answer; }
            })
            .catch(err => { answerDiv.innerHTML = '<b>Ошибка сети:</b> ' + err; });
        }
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE, objects_json=json.dumps(data, ensure_ascii=False))

@app.route('/ask', methods=['POST'])
def ask():
    try:
        req_data = request.get_json()
        question = req_data.get('question')
        context = req_data.get('context')

        system_prompt = "Ты — ГИС-ассистент. Отвечай кратко на русском, используя только эти данные."
        user_prompt = f"Вопрос: {question}\n\nДанные: {json.dumps(context, ensure_ascii=False)}"

        # Обращаемся к Ollama по ЛОКАЛЬНОМУ адресу внутри Colab
        url = "http://127.0.0.1:11434/api/generate"
        payload = {
            "model": "phi3:mini",
            "prompt": f"{system_prompt}\n\n{user_prompt}",
            "stream": False
        }

        response = requests.post(url, json=payload, timeout=300)

        if response.status_code != 200:
            return jsonify({"error": f"Ollama вернула {response.status_code}"})

        result = response.json()
        return jsonify({"answer": result.get('response', 'Нет ответа')})

    except Exception as e:
        return jsonify({"error": str(e)})

def run_app():
    app.run(host='0.0.0.0', port=5000)

threading.Thread(target=run_app).start()
time.sleep(3)

from google.colab.output import eval_js
print("\n" + "="*50)
print("✅ ВАШ САЙТ ГОТОВ! ПЕРЕЙДИТЕ ПО ССЫЛКЕ НИЖЕ:")
print("="*50)
print(eval_js("google.colab.kernel.proxyPort(5000)"))
print("="*50)

Запуск Ollama...


INFO:werkzeug:127.0.0.1 - - [16/Nov/2026 03:55:40] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/Nov/2026 03:55:42] "POST /ask HTTP/1.1" 200 -


Ollama запущена в фоне.
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



✅ ВАШ САЙТ ГОТОВ! ПЕРЕЙДИТЕ ПО ССЫЛКЕ НИЖЕ:
https://5000-m-s-use1b3-hvbzrwxk5f57-b.us-east1.prod.r.colab.dev
